# 🍚 Khám Phá Dữ Liệu & Thử Nghiệm Mô Hình (EDA Notebook)
### Project 17: Phân Loại Hai Giống Gạo (Cammeo vs Osmancik) Từ Hình Dạng Hạt

Notebook này dùng cho mục đích khám phá ban đầu (Exploratory Data Analysis - EDA), kiểm tra tương quan, thử nghiệm trực quan hóa cây quyết định và đối chứng với Rừng ngẫu nhiên.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Nạp và Khảo Sát Dữ Liệu Gốc
- Bộ dữ liệu: `Rice_Cammeo_Osmancik.arff` từ UCI Machine Learning Repository
- Gồm 3,810 mẫu hạt gạo với 7 đặc trưng hình học.

In [ ]:
raw_path = '../data/raw/Rice_Cammeo_Osmancik.arff'
data, meta = arff.loadarff(raw_path)
df = pd.DataFrame(data)
df['Class'] = df['Class'].str.decode('utf-8')

print('Kích thước tập dữ liệu:', df.shape)
df.head()

## 2. Kiểm Tra Chất Lượng Dữ Liệu
- Dữ liệu thiếu (Missing values)
- Dữ liệu trùng lặp (Duplicates)
- Phân bố hai lớp giống gạo (Class balance)

In [ ]:
print('Missing values từng cột:\n', df.isnull().sum())
print('\nSố dòng trùng lặp:', df.duplicated().sum())
print('\nPhân bố lớp:')
display(df['Class'].value_counts())

plt.figure(figsize=(6, 4))
df['Class'].value_counts().plot(kind='bar', color=['lightcoral', 'cornflowerblue'], edgecolor='black')
plt.title('Phân bố số lượng hai giống gạo')
plt.ylabel('Số lượng hạt')
plt.show()

## 3. Thống Kê Mô Tả & Tương Quan Đặc Trưng
- Tính toán trung bình, độ lệch chuẩn, min, max, tứ phân vị cho 7 đặc trưng.
- Khảo sát hệ số tương quan Pearson giữa các đặc trưng.

In [ ]:
features = ['Area', 'Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length', 'Eccentricity', 'Convex_Area', 'Extent']
display(df[features].describe().round(2))

plt.figure(figsize=(9, 7))
corr = df[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Ma trận tương quan giữa 7 đặc trưng hình học')
plt.show()

## 4. Trực Quan Hóa Sự Khác Biệt Giữa 2 Giống Gạo
Khảo sát các đặc trưng có tính phân tách mạnh: `Major_Axis_Length`, `Perimeter`, `Eccentricity`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot chiều dài trục chính
sns.boxplot(x='Class', y='Major_Axis_Length', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Phân bố Chiều Dài Trục Chính (Major_Axis_Length)')

# Scatter phân tán: Major vs Minor Axis
sns.scatterplot(x='Major_Axis_Length', y='Minor_Axis_Length', hue='Class', data=df, alpha=0.6, ax=axes[1], palette='Set1')
axes[1].set_title('Mối liên hệ giữa Trục Chính và Trục Phụ')

plt.tight_layout()
plt.show()

## 5. Thử Nghiệm Huấn Luyện Cây Quyết Định Nhanh
Kiểm tra cây độ sâu nông để xem quy tắc phân loại sơ bộ:

In [ ]:
X = df[features]
y = df['Class']

# Stratified Split 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Cây quyết định nông max_depth=3 để trực quan hóa
tree_demo = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_demo.fit(X_train, y_train)

plt.figure(figsize=(18, 8))
plot_tree(tree_demo, feature_names=features, class_names=tree_demo.classes_, filled=True, rounded=True, fontsize=10)
plt.title('Cây quyết định độ sâu 3 minh họa luật phân loại')
plt.show()

print('Test Accuracy:', tree_demo.score(X_test, y_test).round(4))
print(classification_report(y_test, tree_demo.predict(X_test)))